In [ ]:
%pip install autogluon

In [ ]:
import numpy as np
import pandas as pd

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
from autogluon.tabular import TabularDataset, TabularPredictor
from autogluon.common import space

In [ ]:
# Load dataset into a TabularDataset
data = TabularDataset('/kaggle/input/zelestra/train.csv')

In [ ]:
test = TabularDataset('/kaggle/input/zelestra/test.csv')

In [ ]:
predictor = TabularPredictor(
    label='efficiency',
    problem_type='regression',
    eval_metric='root_mean_squared_error',
).fit(
    train_data=data,
    time_limit=7200,
    presets='best_quality',
    hyperparameters={
        'XGB': {
            'n_estimators': space.Int(100, 1000),
            'max_depth': space.Int(3, 15),
            'learning_rate': space.Real(1e-4, 0.1, log=True),
            'subsample': space.Real(0.5, 1.0),
            'colsample_bytree': space.Real(0.5, 1.0),
            'gamma': space.Real(0, 5),
            'min_child_weight': space.Int(1, 10),
            'reg_alpha': space.Real(0.0, 5.0),
            'reg_lambda': space.Real(0.0, 5.0),
            'scale_pos_weight': space.Real(1.0, 10.0),
        },
        'CAT': {
            'iterations': space.Int(100, 1000),
            'depth': space.Int(3, 10),
            'learning_rate': space.Real(0.001, 0.1, log=True),
            'l2_leaf_reg': space.Real(1, 10),
            'bagging_temperature': space.Real(0, 1),
            'border_count': space.Int(32, 255),
        },
        'GBM': {  # LightGBM
            'num_leaves': space.Int(20, 150),
            'max_depth': space.Int(3, 15),
            'learning_rate': space.Real(1e-4, 0.1, log=True),
            'n_estimators': space.Int(100, 1000),
            'min_child_weight': space.Int(1, 10),
            'subsample': space.Real(0.5, 1.0),
            'colsample_bytree': space.Real(0.5, 1.0),
            'reg_alpha': space.Real(0.0, 5.0),
            'reg_lambda': space.Real(0.0, 5.0),
        },
        'LR': {  # ElasticNet
            'penalty': 'elasticnet',
            'l1_ratio': space.Real(0.0, 1.0),  # 0 = Ridge, 1 = Lasso
            'alpha': space.Real(1e-5, 1.0, log=True),  # Regularization strength
            'max_iter': 1000,
        }
    },
    hyperparameter_tune_kwargs={
        'num_trials': 200,
        'scheduler': 'local',
        'searcher': 'random',
    }
)


In [ ]:
leaderboard = predictor.leaderboard(data, silent=True)
print(leaderboard)

In [ ]:
predictions = predictor.predict(test)

In [ ]:
test

In [ ]:
predictions

In [ ]:
sample = pd.read_csv('/kaggle/input/playground-series-s4e9/sample_submission.csv')

In [ ]:
sample.head()

In [ ]:
id = test.pop('id')

# Create a submission DataFrame
submission_df = pd.DataFrame({
    'id': id,
    'class': predictions
})

# Save the submission DataFrame to a CSV file
submission_df.to_csv('submission.csv', index=False)
print("Submission file created: submission.csv")

In [ ]:
submission = pd.read_csv('/kaggle/working/submission.csv')
submission.head()